# Xarray-Spatial Diffusion: Scalar field spreading

Heat, moisture, and contaminants all spread through space over time. The `diffuse()` function solves the 2D diffusion equation on a raster, stepping a scalar field forward with an explicit finite-difference scheme. It handles uniform and spatially varying diffusivity, four boundary modes, and runs on all four xrspatial backends.

### What you'll build

1. Spread a hot spot across a grid and watch the field evolve
2. Compare boundary modes side by side
3. Block heat flow with a spatially varying diffusivity wall
4. Simulate heat spreading through a building floor plan after an HVAC failure

![Diffusion preview](images/diffusion_hvac_preview.png)

[Point source diffusion](#Point-source-diffusion) · [Boundary modes](#Boundary-modes) · [Spatially varying diffusivity](#Spatially-varying-diffusivity) · [HVAC failure simulation](#HVAC-failure-simulation)

Standard imports plus the `diffuse` function from `xrspatial.diffusion`.

In [ ]:
%matplotlib inline
import numpy as np
import xarray as xr

import matplotlib.pyplot as plt

from xrspatial.diffusion import diffuse

## Point source diffusion

The simplest test case: a Gaussian hot spot in the center of a cold field. The [diffusion equation](https://en.wikipedia.org/wiki/Diffusion_equation) spreads that energy outward, broadening the peak and lowering the maximum temperature. Running more steps lets the heat spread further.

Four panels show the field at steps 0, 10, 50, and 200. Early steps clip at the shared color scale — the peak is higher than `vmax` — while later steps show the full shape of the broadened distribution.

In [ ]:
# 51x51 grid with a Gaussian hot spot at the center
shape = (51, 51)
yy, xx = np.mgrid[0:shape[0], 0:shape[1]]
data = 25.0 * np.exp(-((yy - 25)**2 + (xx - 25)**2) / (2 * 2**2))

initial = xr.DataArray(data, dims=['y', 'x'], attrs={'res': (1.0, 1.0)})

steps_list = [0, 10, 50, 200]
results = {}
for s in steps_list:
    if s == 0:
        results[s] = initial
    else:
        results[s] = diffuse(initial, diffusivity=1.0, steps=s, boundary='nearest')

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, s in zip(axes, steps_list):
    results[s].plot.imshow(ax=ax, cmap='inferno', vmin=0, vmax=4,
                           add_colorbar=False)
    ax.set_title(f'Step {s}', fontsize=12)
    ax.set_axis_off()
fig.subplots_adjust(right=0.90)
cbar_ax = fig.add_axes([0.91, 0.15, 0.015, 0.7])
fig.colorbar(axes[-1].images[-1], cax=cbar_ax, label='Temperature')

At step 0 the energy is concentrated in a tight Gaussian (saturated in the shared color scale). By step 200 it has spread into a broad, low mound. Total heat is conserved with `boundary='nearest'`, just redistributed across the grid.

## Boundary modes

The `boundary` parameter controls what happens when the stencil reaches the edge of the raster. Four options are available:

- **`'nan'`**: pad with NaN, so edges decay to NaN over time
- **`'nearest'`**: repeat edge values, conserving energy
- **`'reflect'`**: mirror the interior, simulating a symmetric boundary (equivalent to `'nearest'` for the 5-point stencil)
- **`'wrap'`**: connect opposite edges, making the domain periodic

A broad Gaussian heat source placed near the top edge makes the differences obvious. Three panels below show `nan`, `nearest`, and `wrap` after 12 diffusion steps. The `nan` mode loses heat at all four edges as NaN propagates inward, while `nearest` reflects heat back and `wrap` sends it to the opposite side of the grid.

In [ ]:
# Broad Gaussian near the top edge (high y = top of displayed image)
grid = 51
yy, xx = np.mgrid[0:grid, 0:grid]
data_edge = 50.0 * np.exp(-((yy - 46)**2 + (xx - 25)**2) / (2 * 6**2))

edge_field = xr.DataArray(data_edge, dims=['y', 'x'], attrs={'res': (1.0, 1.0)})

boundaries = ['nan', 'nearest', 'wrap']

fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))
vmax = 45  # shared scale across panels
for ax, bnd in zip(axes, boundaries):
    r = diffuse(edge_field, diffusivity=1.0, steps=12, boundary=bnd)
    r.plot.imshow(ax=ax, cmap='inferno', vmin=0, vmax=vmax, add_colorbar=False)
    ax.set_title(f"boundary='{bnd}'", fontsize=11)
    ax.set_axis_off()
fig.subplots_adjust(right=0.90)
cbar_ax = fig.add_axes([0.91, 0.15, 0.015, 0.7])
fig.colorbar(axes[-1].images[-1], cax=cbar_ax, label='Temperature')

<div class="alert alert-block alert-warning">
<b>Boundary choice affects conservation.</b> With <code>boundary='nan'</code>, heat that reaches the edge disappears — and NaN propagates one cell inward per step, so the usable domain shrinks over time. If you need total energy to stay constant (mass balance, pollutant budgets), use <code>'nearest'</code> or <code>'wrap'</code> instead.
</div>

## Spatially varying diffusivity

Passing a DataArray for the `diffusivity` parameter lets you model materials with different thermal properties. A region of low diffusivity acts as an insulating wall that slows heat transfer.

Below, a vertical strip of near-zero diffusivity (0.01) splits the grid in half. A heat source on the left side runs for 300 steps. The left panel shows the diffusivity field and the right panel shows the resulting temperature.

In [ ]:
shape = (51, 51)
yy, xx = np.mgrid[0:shape[0], 0:shape[1]]
data = 25.0 * np.exp(-((yy - 25)**2 + (xx - 10)**2) / (2 * 2**2))

# Diffusivity: uniform 1.0 except a thin vertical wall
alpha = np.ones(shape)
alpha[:, 25] = 0.01

field = xr.DataArray(data, dims=['y', 'x'], attrs={'res': (1.0, 1.0)})
alpha_da = xr.DataArray(alpha, dims=['y', 'x'], attrs={'res': (1.0, 1.0)})

result = diffuse(field, diffusivity=alpha_da, steps=300, boundary='nearest')

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

alpha_da.plot.imshow(ax=axes[0], cmap='gray', add_colorbar=True,
                     cbar_kwargs={'label': 'Diffusivity', 'shrink': 0.7})
axes[0].set_title('Diffusivity field', fontsize=12)
axes[0].set_axis_off()

result.plot.imshow(ax=axes[1], cmap='inferno', vmin=0, add_colorbar=True,
                   cbar_kwargs={'label': 'Temperature', 'shrink': 0.7})
axes[1].set_title('Temperature after 300 steps', fontsize=12)
axes[1].set_axis_off()

plt.tight_layout()

Most of the heat stays on the left side. The wall does not block transfer completely (diffusivity is 0.01, not zero), so a small amount leaks through. Setting a column to NaN would create a true barrier instead.

## HVAC failure simulation

A more practical scenario: a building floor plan where the cooling system fails. The corridor starts at 20 °C. The enclosed room jumps to 50 °C when the chiller goes offline. Walls are modeled as near-zero diffusivity (0.001), blocking heat transfer. A door opening in the top wall lets heat escape into the corridor above.

Four snapshots show how the hot room cools as heat bleeds out the doorway.

In [ ]:
shape = (61, 61)
temp = np.full(shape, 20.0)

# Room interior heated to 50 °C after HVAC failure
temp[21:40, 11:50] = 150.0

# Spatially varying diffusivity: walls block heat, door lets it through
alpha = np.full(shape, 0.5)
alpha[20, 10:50] = 0.001   # bottom wall
alpha[40, 10:50] = 0.001   # top wall
alpha[20:41, 10] = 0.001   # left wall
alpha[20:41, 50] = 0.001   # right wall
alpha[40, 29:32] = 0.5     # door opening in top wall

floor_plan = xr.DataArray(temp, dims=['y', 'x'], attrs={'res': (1.0, 1.0)})
alpha_da = xr.DataArray(alpha, dims=['y', 'x'], attrs={'res': (1.0, 1.0)})

# Mask wall cells for display (render as gaps)
wall_mask = alpha_da.values < 0.01

snapshots = [0, 30, 80, 200, 500]
fig, axes = plt.subplots(1, 5, figsize=(18, 4.5))
for ax, s in zip(axes, snapshots):
    if s == 0:
        r = floor_plan.copy()
    else:
        r = diffuse(floor_plan, diffusivity=alpha_da, steps=s, boundary='nearest')
    display = r.copy()
    display.values[wall_mask] = np.nan
    im = display.plot.imshow(ax=ax, cmap='coolwarm', vmin=20, vmax=50,
                             add_colorbar=False)
    ax.set_title(f'Step {s}', fontsize=12)
    ax.set_axis_off()
fig.subplots_adjust(right=0.92)
cbar_ax = fig.add_axes([0.93, 0.15, 0.015, 0.7])
fig.colorbar(im, cax=cbar_ax, label='Temperature (°C)')
import pathlib
pathlib.Path('images').mkdir(exist_ok=True)
fig.savefig('images/diffusion_hvac_preview.png', bbox_inches='tight', dpi=120)

By step 200 the room has cooled substantially, and a plume of heat is leaking through the doorway. The low-diffusivity walls block nearly all heat transfer, so the only significant path out is through the door opening.

<div class="alert alert-block alert-warning">
<b>Time step stability.</b> The explicit forward-Euler scheme is conditionally stable. When you omit <code>dt</code>, the solver picks the largest stable step automatically (<code>dt = 0.25 * dx^2 / max(alpha)</code>). If you set <code>dt</code> manually and make it too large, the solution will oscillate and blow up. Stick with the auto value unless you have a specific reason to override it.
</div>

### References

- [Diffusion equation](https://en.wikipedia.org/wiki/Diffusion_equation), Wikipedia
- [Forward Euler method](https://en.wikipedia.org/wiki/Euler_method), Wikipedia
- [Finite difference method for heat equation](https://en.wikipedia.org/wiki/Finite_difference_method#Example:_The_heat_equation), Wikipedia
- [CFL stability condition](https://en.wikipedia.org/wiki/Courant%E2%80%93Friedrichs%E2%80%93Lewy_condition), Wikipedia